<h1>Table of Contents<span class="tocSkip"></span></h1>
<div class="toc"><ul class="toc-item"><li><span><a href="#Var-Reduction-Speed" data-toc-modified-id="Var-Reduction-Speed-1"><span class="toc-item-num">1&nbsp;&nbsp;</span>Var Reduction Speed</a></span></li><li><span><a href="#Var-Reduction" data-toc-modified-id="Var-Reduction-2"><span class="toc-item-num">2&nbsp;&nbsp;</span>Var Reduction</a></span><ul class="toc-item"><li><span><a href="#Dimensional-wise-Test-Taker-Percentage" data-toc-modified-id="Dimensional-wise-Test-Taker-Percentage-2.1"><span class="toc-item-num">2.1&nbsp;&nbsp;</span>Dimensional-wise Test Taker Percentage</a></span></li></ul></li><li><span><a href="#Posterior-Mean-v.s-Oracle-Mean" data-toc-modified-id="Posterior-Mean-v.s-Oracle-Mean-3"><span class="toc-item-num">3&nbsp;&nbsp;</span>Posterior Mean v.s Oracle Mean</a></span></li><li><span><a href="#Posterior-Median-v.s-Oracle-Median" data-toc-modified-id="Posterior-Median-v.s-Oracle-Median-4"><span class="toc-item-num">4&nbsp;&nbsp;</span>Posterior Median v.s Oracle Median</a></span></li><li><span><a href="#Posterior-First-Quantile-v.s-Oracle-First-Quantile" data-toc-modified-id="Posterior-First-Quantile-v.s-Oracle-First-Quantile-5"><span class="toc-item-num">5&nbsp;&nbsp;</span>Posterior First Quantile v.s Oracle First Quantile</a></span></li><li><span><a href="#Posterior-Third-Quantile-v.s-Oracle-Third-Quantile" data-toc-modified-id="Posterior-Third-Quantile-v.s-Oracle-Third-Quantile-6"><span class="toc-item-num">6&nbsp;&nbsp;</span>Posterior Third Quantile v.s Oracle Third Quantile</a></span></li><li><span><a href="#Training-Dynamics" data-toc-modified-id="Training-Dynamics-7"><span class="toc-item-num">7&nbsp;&nbsp;</span>Training Dynamics</a></span></li><li><span><a href="#Correlation-between-oracle-and-the-estimates-after-20-items?" data-toc-modified-id="Correlation-between-oracle-and-the-estimates-after-20-items?-8"><span class="toc-item-num">8&nbsp;&nbsp;</span>Correlation between oracle and the estimates after 20 items?</a></span></li><li><span><a href="#Factor-Loading-Matrix-Visualization" data-toc-modified-id="Factor-Loading-Matrix-Visualization-9"><span class="toc-item-num">9&nbsp;&nbsp;</span>Factor Loading Matrix Visualization</a></span></li></ul></div>

In [1]:
import warnings
warnings.filterwarnings('ignore', category=UserWarning)
import pickle
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pathlib
import yaml
import argparse
import pandas as pd
import bayesian_cat as bcat
sns.set()

In [ ]:
model_dir = pathlib.Path.home().joinpath(pathlib.Path("bayesian-cat", "models", "cat_cog_models"))
model_filenames = ["COG_s02_kl_eap_51.pickle", "COG_s02_kl_pos_51.pickle", "COG_s02_mi_sir_51.pickle", "COG_s02_predictive_variance_e_51.pickle"]
# previous baseline model
model_keys = ["EAP","Max Pos", "MI", "Max Var"]
models = {}
for i, filename in enumerate(model_filenames):
    with open(model_dir.joinpath(filename), "rb") as input_file:
        models[model_keys[i]] = pickle.load(input_file)
# current deep q learning
q_cat_dir = model_dir.joinpath("Q-CAT")
Q_model_filenames = ["x08_s05_QCAT_51_online-Q-network-v4-6-factor_NN_bifactor_alphas_150000_first0-1_detailed_v4_1.5e-05_128_30000_600000_0.14_fn_69000.pt_final.pickle"]
Q_learning_keys = ["Q-Learning-0.14"]
model_keys += Q_learning_keys
for i, filename in enumerate(Q_model_filenames):
    with open(q_cat_dir.joinpath(filename), "rb") as input_file:
        models[Q_learning_keys[i]] = pickle.load(input_file)

# read params and data
loadings_dir = model_dir.joinpath("bifactor_alphas.feather")
factors_dir = model_dir.joinpath("latent_traits_abs_right.feather")
response_dir = model_dir.joinpath("binary_response_abs_right.feather")
loading_df = pd.read_feather(loadings_dir)
factor_df = pd.read_feather(factors_dir)
response_df = pd.read_feather(response_dir)

        
n, max_items=response_df.values.shape[0], 51
num_models = len(model_keys)
model = models["MI"]
m,k = model.m, model.k
true_thetas = model.true_thetas
alphas = model.alphas
intercepts = model.intercepts

In [ ]:
# Oracle Distribution
# params = {"thetas": true_thetas, "alphas": alphas, "intercepts": intercepts}
# oracle = bcat.OracleCAT(response_df.values, params)
# bayes_oracle = oracle.sun_inference(1000, return_samples=True)
# bayes_oracle["response"] = response_df.values
# with open('new_bayesian_oracle_exact_abs_right_51.pickle', 'wb') as handle:
#      pickle.dump(bayes_oracle, handle, protocol=pickle.HIGHEST_PROTOCOL)
with open('new_bayesian_oracle_exact_abs_right_51.pickle', 'rb') as handle:
    bayes_oracle = pickle.load(handle)

In [ ]:
true_samples= bayes_oracle["samples"][:n]
true_quantiles = {0.25:np.zeros((n, k)), 0.5:np.zeros((n, k)), 0.75 :np.zeros((n, k)), "oracle_mean": np.zeros((n,k))} # keys: 1, 2, 3 Vals: [],[],[]
for i in range(n):
    temp_sample = true_samples[i]
    for quantile in true_quantiles.keys():
        if quantile == "oracle_mean":
            true_quantiles[quantile][i]= np.mean(temp_sample, axis=0)
        else:
            true_quantiles[quantile][i] = np.quantile(temp_sample, quantile, axis=0)

# Var Reduction Speed

In [ ]:
# Record Quantile result
#Quantile_result key is model, value is a disctionary: with key as 
# quantile_result = {}
# for model_name, model in models.items():
#     print(model_name)
#     model_quantiles = {}
#     max_var_array = np.zeros((n, max_items))
#     max_mse_array = np.zeros((n, max_items))
#     if model_name[0] != "Q":
#         d1_tensor = model.tts.d1_tensor[:n, :max_items, :]
#         d2_array = model.tts.d2_array[:n, :max_items]
#         s_array = model.tts.s_array[:n, :max_items]
#     else:
#         d1_tensor = model.tts.d1_tensor[:n, 1:(max_items+1), :]
#         d2_array = model.tts.d2_array[:n, 1:(max_items+1)]
#         s_array = model.tts.s_array[:n, 1:(max_items+1)]
#     for j in range(max_items):
#         model_quantiles[j] = {0.25:np.zeros((n, k)), 0.5:np.zeros((n, k)), 0.75 :np.zeros((n, k)),
#                              "var": np.zeros((n,k))}
#         model_samples = bcat.sample_from_sun_all(1000, d1_tensor[:,:(j+1),:], d2_array[:, :(j+1)],  s_array[:, :(j+1)], num_workers =8)
#         model_quantiles[j]["var"] = np.var(model_samples, axis=1)
#         for quantile in true_quantiles.keys():
#             if quantile == "oracle_mean":
#                 model_quantiles[j][quantile] = np.mean(model_samples, axis=1)
#             else:
#                 model_quantiles[j][quantile] = np.quantile(model_samples, quantile, axis=1)
#     quantile_result[model_name] = model_quantiles

# with open("new_quantile_result_abs_right_51.pickle", 'wb') as handle:
#     pickle.dump(quantile_result, handle, protocol=pickle.HIGHEST_PROTOCOL)
    
with open('new_quantile_result_abs_right_51.pickle', 'rb') as handle:
     quantile_result= pickle.load(handle)

# Var Reduction

## Dimensional-wise Test Taker Percentage

In [ ]:
dim_names = ["Primary Factor", "Executive Function", "Episodic Memory", "Processing Speed", "Semantic Memory", "Working Memory" ]

In [ ]:
x_vals = np.arange(1,max_items+1,1)
test_taker_percentage = {}
fig, axs = plt.subplots(2, 3, figsize=(15, 16))
threshold = 0.16
for dim in range(k):
    coord_0 = dim//3
    coord_1 = dim % 3
    for model_name, model in models.items():
        percentage = np.zeros(max_items)
        for h in range(max_items):
            percentage[h] = np.mean(quantile_result[model_name][h]["var"][:, 0] < threshold) * 100
        axs[coord_0][coord_1].plot(x_vals, percentage, label=model_name)
    axs[coord_0][coord_1].set_xlabel("Number of items")
    axs[coord_0][coord_1].set_ylabel("Test Termination Rate")
    axs[coord_0][coord_1].legend()
    axs[coord_0][coord_1].set_title("{}: Test Termination Rate".format(dim_names[dim]))


In [ ]:
# x_vals = np.arange(1,max_items+1,1)
# plt.figure(figsize=(7,5))
# threshold = 0.16
# for model_name, model in models.items():
#     percentage = np.zeros(max_items)
#     for h in range(max_items):
#         #percentage[h] = np.mean(np.max(quantile_result[model_name][h]["var"], axis=1) < threshold) * 100
#         percentage[h] = np.mean(quantile_result[model_name][h]["var"][:, 0] < threshold) * 100
#     plt.plot(x_vals, percentage, label=model_name)
    
# plt.legend()
# plt.title("Number of Items v.s MCAT Completion Rate")
# plt.ylabel("Test Completion Rate Percentage")
# plt.xlabel("Number of Administered Items")

In [ ]:
x_vals = np.arange(1,max_items+1,1)
plt.figure(figsize=(7,5))
threshold = 0.16
for model_name, model in models.items():
    nonterminated_set = set(np.arange(0,n,1))
    percentage = np.zeros(max_items)
    for h in range(max_items):
        #cur_nonterminated = np.where(np.max(quantile_result[model_name][h]["var"], axis=1) > threshold)[0]
        cur_nonterminated = np.where(quantile_result[model_name][h]["var"][:, 0] > threshold)[0]
        nonterminated_set = nonterminated_set.intersection(cur_nonterminated)
        percentage[h] = (n-len(nonterminated_set))/n
    if model_name[0] == "Q":
        model_name = "Q-learning"
    plt.plot(x_vals, percentage, label=model_name)
    
plt.legend()
plt.title("Number of Items v.s Cumulative Percentage of Completed Test Sessions")
plt.ylabel("Cumulative Percentage of Completed Test Sessions")
plt.xlabel("Number of Administered Items")


In [ ]:
x_vals = np.arange(1,max_items+1,1)
plt.figure(figsize=(7,5))
threshold = 0.09
for model_name, model in models.items():
    nonterminated_set = set(np.arange(0,n,1))
    percentage = np.zeros(max_items)
    for h in range(max_items):
        #cur_nonterminated = np.where(np.max(quantile_result[model_name][h]["var"], axis=1) > threshold)[0]
        cur_nonterminated = np.where(quantile_result[model_name][h]["var"][:, 0] > threshold)[0]
        nonterminated_set = nonterminated_set.intersection(cur_nonterminated)
        percentage[h] = (n-len(nonterminated_set))/n
    if model_name[0] == "Q":
        model_name = "Q-learning"
    plt.plot(x_vals, percentage, label=model_name)
    
plt.legend()
plt.title("Number of Items v.s Cumulative Percentage of Completed Test Sessions")
plt.ylabel("Cumulative Percentage of Completed Test Sessions")
plt.xlabel("Number of Administered Items")


In [ ]:
# Ensure figure is explicitly created
fig, ax = plt.subplots(figsize=(7,5))

threshold = 0.16
for model_name, model in models.items():
    nonterminated_set = set(np.arange(0, n, 1))
    percentage = np.zeros(max_items)
    
    for h in range(max_items):
        cur_nonterminated = np.where(quantile_result[model_name][h]["var"][:, 0] > threshold)[0]
        nonterminated_set = nonterminated_set.intersection(cur_nonterminated)
        percentage[h] = (n - len(nonterminated_set)) / n
    if model_name[0] == "Q":
        model_name = "Q-Learning"
    ax.plot(x_vals, percentage, label=model_name)

# Labels and legend
ax.set_title("Number of Items vs. Cumulative Percentage of Completed Test Sessions", fontsize=12)
ax.set_xlabel("Number of Administered Items", fontsize=12)
ax.set_ylabel("Cumulative Percentage of Completed Test Sessions", fontsize=12)
ax.legend(fontsize=10)

# Save the figure BEFORE plt.show()
plt.savefig("cog_completion_rate.pdf", format="pdf", dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
model_names = list(models.keys())
average_termination = {}
for model_name, model in models.items():
    count = 0
    nonterminated_set = set(np.arange(0,n,1))
    for h in range(max_items):
        #cur_nonterminated = np.where(np.max(quantile_result[model_name][h]["var"], axis=1) > threshold)[0]
        cur_nonterminated = np.where(quantile_result[model_name][h]["var"][:, 0] > threshold)[0]
        nonterminated_set = nonterminated_set.intersection(cur_nonterminated)
        count+= len(nonterminated_set)
    average_termination[model_name] = count/n

In [ ]:
average_termination

# Posterior Mean v.s Oracle Mean

In [ ]:
x_vals = np.arange(1,max_items+1,1)
test_taker_percentage = {}
fig, axs = plt.subplots(2, 3, figsize=(20, 20))
for dim in range(k):
    coord_0 = dim//3
    coord_1 = dim % 3
    for model_name, model in models.items():
        mse = np.zeros(max_items)
        for h in range(max_items):
            mse[h] = np.mean(np.square(quantile_result[model_name][h]["oracle_mean"][:, dim] - true_quantiles["oracle_mean"][:, dim]))
        axs[coord_0][coord_1].plot(x_vals, mse, label=model_name)
    axs[coord_0][coord_1].set_xlabel("Number of items")
    axs[coord_0][coord_1].set_ylabel("MSE Between Posterior Mean and Oracle Mean")
    axs[coord_0][coord_1].legend()
    axs[coord_0][coord_1].set_title("{}: Number of Items v.s Estimation MSE".format(dim_names[dim]))


In [ ]:
x_vals = np.arange(1,max_items+1,1)
test_taker_percentage = {}
fig, ax = plt.subplots(figsize=(7, 5))
for model_name, model in models.items():
    mse = np.zeros(max_items)
    for h in range(max_items):
        mse[h] = np.mean(np.square(quantile_result[model_name][h]["oracle_mean"][:, 0] - true_quantiles["oracle_mean"][:, 0]))
    if model_name[0] == "Q":
        model_name = "Q-learning"
    ax.plot(x_vals, mse, label=model_name)


# Labels and legend
ax.set_title("MSE Between Posterior Mean and Oracle Mean in Primary Dimension", fontsize=12)
ax.set_xlabel("Number of Administered Items", fontsize=12)
ax.set_ylabel("MSE Between Posterior Mean and Oracle Mean", fontsize=12)
ax.legend(fontsize=10)

# Save the figure BEFORE plt.show()
plt.savefig("cog_completion_mse.pdf", format="pdf", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
win_ter_idx = 19
winshare_tables = np.zeros((len(models.keys()),k))
model_names = list(models.keys())
for dim in range(k):
    mse_table = np.zeros((n, len(model_names)))
    for i, model_name in enumerate(model_names):
        mse_table[:, i] = np.square(quantile_result[model_name][win_ter_idx]["oracle_mean"][:, dim] -  true_quantiles["oracle_mean"][:, dim])
    winners = np.argmin(mse_table, axis=1)
    winshare_columns = np.zeros(len(model_names))
    for i, model_name in enumerate(model_names):
        winshare_columns[i] = np.sum(winners==i)/n
    winshare_tables[:, dim] = winshare_columns
        
winshare_tables= pd.DataFrame(winshare_tables, columns = ["dim"+str(dim) for dim in range(k)])
winshare_tables.index = model_names
winshare_tables

In [ ]:
winshare_tables["avg_termination"] = winshare_tables.index.map(average_termination) 

In [ ]:
#winshare_tables["selection_time"] = winshare_tables.index.map(average_termination) 

In [ ]:
total_items = 50*730
selection_time = {"EAP":799/total_items, "Max Pos":1175/total_items, "MI": 1481/total_items , "Max Var": 811/total_items, "Q-Learning-0.14":916/total_items}

In [ ]:
winshare_tables["selection_time"] = winshare_tables.index.map(selection_time) 
winshare_tables

In [ ]:
winshare_tables.columns = dim_names+["avg_termination", "selection_time"]

In [ ]:
winshare_tables = winshare_tables[["avg_termination"]+ dim_names + ["selection_time"]]

In [ ]:
winshare_tables.transpose()

In [ ]:
# from scipy.stats import spearmanr

# win_ter_idx = 29
# roc_tables = np.zeros((len(models.keys()),k))
# model_names = list(models.keys())
# for dim in range(k):
#     for i, model_name in enumerate(model_names):
#         roc_tables[i, dim]=spearmanr(quantile_result[model_name][win_ter_idx]["oracle_mean"][:, dim], true_quantiles["oracle_mean"][:, dim])[0]
    
    
# roc_tables= pd.DataFrame(roc_tables, columns = ["dim"+str(dim) for dim in range(k)])
# roc_tables.index = model_names
# roc_tables

# Posterior Median v.s Oracle Median

In [ ]:
x_vals = np.arange(1,max_items+1,1)
test_taker_percentage = {}
fig, axs = plt.subplots(2, 3, figsize=(20, 20))
for dim in range(k):
    coord_0 = dim//3
    coord_1 = dim % 3
    for model_name, model in models.items():
        mse = np.zeros(max_items)
        for h in range(max_items):
            mse[h] = np.mean(np.square(quantile_result[model_name][h][0.5][:, dim] - true_quantiles[0.5][:, dim]))
        axs[coord_0][coord_1].plot(x_vals, mse, label=model_name)
    axs[coord_0][coord_1].set_xlabel("Number of items")
    axs[coord_0][coord_1].set_ylabel("MSE Between Posterior Median and Oracle Median")
    axs[coord_0][coord_1].legend()
    axs[coord_0][coord_1].set_title("{}: Number of Items v.s Estimation MSE".format(dim_names[dim]))


In [ ]:
winshare_tables = np.zeros((len(models.keys()),k))
model_names = list(models.keys())
for dim in range(k):
    mse_table = np.zeros((n, len(model_names)))
    for i, model_name in enumerate(model_names):
        mse_table[:, i] = np.square(quantile_result[model_name][win_ter_idx][0.5][:, dim] -  true_quantiles[0.5][:, dim])
    winners = np.argmin(mse_table, axis=1)
    winshare_columns = np.zeros(len(model_names))
    for i, model_name in enumerate(model_names):
        winshare_columns[i] = np.sum(winners==i)/n
    winshare_tables[:, dim] = winshare_columns
        
winshare_tables= pd.DataFrame(winshare_tables, columns = ["dim"+str(dim) for dim in range(k)])
winshare_tables.index = model_names
winshare_tables

# Posterior First Quantile v.s Oracle First Quantile

In [ ]:
x_vals = np.arange(1,max_items+1,1)
test_taker_percentage = {}
fig, axs = plt.subplots(2, 3, figsize=(20, 20))
for dim in range(k):
    coord_0 = dim//3
    coord_1 = dim % 3
    for model_name, model in models.items():
        mse = np.zeros(max_items)
        for h in range(max_items):
            mse[h] = np.mean(np.square(quantile_result[model_name][h][0.25][:, dim] - true_quantiles[0.25][:, dim]))
        axs[coord_0][coord_1].plot(x_vals, mse, label=model_name)
    axs[coord_0][coord_1].set_xlabel("Number of items")
    axs[coord_0][coord_1].set_ylabel("MSE Between Posterior First Quantile and Oracle First Quantile")
    axs[coord_0][coord_1].legend()
    axs[coord_0][coord_1].set_title("{}: Number of Items v.s Estimation MSE".format(dim_names[dim]))


In [ ]:
winshare_tables = np.zeros((len(models.keys()),k))
model_names = list(models.keys())
for dim in range(k):
    mse_table = np.zeros((n, len(model_names)))
    for i, model_name in enumerate(model_names):
        mse_table[:, i] = np.square(quantile_result[model_name][win_ter_idx][0.25][:, dim] -  true_quantiles[0.25][:, dim])
    winners = np.argmin(mse_table, axis=1)
    winshare_columns = np.zeros(len(model_names))
    for i, model_name in enumerate(model_names):
        winshare_columns[i] = np.sum(winners==i)/n
    winshare_tables[:, dim] = winshare_columns
        
winshare_tables= pd.DataFrame(winshare_tables, columns = ["dim"+str(dim) for dim in range(k)])
winshare_tables.index = model_names
winshare_tables

# Posterior Third Quantile v.s Oracle Third Quantile

In [ ]:
x_vals = np.arange(1,max_items+1,1)
test_taker_percentage = {}
fig, axs = plt.subplots(2, 3, figsize=(20, 20))
for dim in range(k):
    coord_0 = dim//3
    coord_1 = dim % 3
    for model_name, model in models.items():
        mse = np.zeros(max_items)
        for h in range(max_items):
            mse[h] = np.mean(np.square(quantile_result[model_name][h][0.75][:, dim] - true_quantiles[0.75][:, dim]))
        axs[coord_0][coord_1].plot(x_vals, mse, label=model_name)
    axs[coord_0][coord_1].set_xlabel("Number of items")
    axs[coord_0][coord_1].set_ylabel("MSE Between Posterior Third Quantile and Oracle Third Quantile")
    axs[coord_0][coord_1].legend()
    axs[coord_0][coord_1].set_title("{}: Number of Items v.s Estimation MSE".format(dim_names[dim]))


In [ ]:
winshare_tables = np.zeros((len(models.keys()),k))
model_names = list(models.keys())
for dim in range(k):
    mse_table = np.zeros((n, len(model_names)))
    for i, model_name in enumerate(model_names):
        mse_table[:, i] = np.square(quantile_result[model_name][win_ter_idx][0.75][:, dim] -  true_quantiles[0.75][:, dim])
    winners = np.argmin(mse_table, axis=1)
    winshare_columns = np.zeros(len(model_names))
    for i, model_name in enumerate(model_names):
        winshare_columns[i] = np.sum(winners==i)/n
    winshare_tables[:, dim] = winshare_columns
        
winshare_tables= pd.DataFrame(winshare_tables, columns = ["dim"+str(dim) for dim in range(k)])
winshare_tables.index = model_names
winshare_tables

# Training Dynamics

In [ ]:
with open("detailed_log.pickle", "rb") as input_file:
    training_log = pickle.load(input_file)
episode_rewards = np.array([training_log[i]["learned_reward"] for i in range(40001)])
reward_period = 500
plot_length = episode_rewards.shape[0]//reward_period
average_rewards = np.zeros(plot_length)
for i in range(plot_length):
    average_rewards[i] = np.mean(episode_rewards[(i*reward_period):((i+1)*reward_period)])

In [ ]:
episode_loss = np.array([training_log[i]["learned_loss"] for i in range(40001)])
reward_period = 500
plot_length = episode_rewards.shape[0]//reward_period
average_loss = np.zeros(plot_length)
for i in range(plot_length):
    average_loss[i] = np.mean(episode_loss[(i*reward_period):((i+1)*reward_period)])

In [ ]:
x_vals = np.arange(0, 40000, 500)
fig, axs = plt.subplots(1, 2, figsize=(20, 7))
axs[0].plot(x_vals, average_rewards)
axs[0].set_xlabel("Number of Episodes")
axs[0].set_ylabel("Average Rewards Over Every 500 Episodes")
axs[0].set_title("Increase of Rewards During Double Q-Learning")
axs[1].plot(x_vals, average_loss)
axs[1].set_xlabel("Number of Episodes")
axs[1].set_ylabel("Average Validation Loss Over Every 500 Episodes")
axs[1].set_title("Decrease of Validation Loss During Double Q-Learning")
# Save the figure to a PDF file
plt.savefig("training_dynamics_cat_cog.pdf", format="pdf")

# Show the plot
plt.show()

# Correlation between oracle and the estimates after 20 items?

In [ ]:
true_oracle_mean =  true_quantiles["oracle_mean"][:, 0]

In [ ]:
win_ter_idx = 19 
for model_name in model_names:
    print(model_name)
    correlation = np.corrcoef(quantile_result[model_name][win_ter_idx]["oracle_mean"][:, 0], true_oracle_mean)[0, 1]
    print(correlation)

# Factor Loading Matrix Visualization

In [ ]:
response_df

In [ ]:
# Create the plot
# Example n x 4 array (mostly zeros)
all_items = response_df.columns

num_items= len(all_items)  # Number of rows


# Row labels (example labels)
row_labels = all_items
col_labels = dim_names

fig, ax = plt.subplots(figsize=(8, 8))
factor_loadings = alphas / np.sqrt(1 + np.sum(alphas**2, axis=1, keepdims=True))
cax = ax.imshow(factor_loadings, cmap='coolwarm', aspect='auto', interpolation='none')  # No interpolation to remove white gaps

# Add color bar
cbar = plt.colorbar(cax, ax=ax)
cbar.set_label('Value Scale')

# Set row and column labels
ax.set_yticks(np.arange(num_items))
ax.set_yticklabels(row_labels, fontsize=6)

ax.set_xticks(np.arange(6))
ax.set_xticklabels(col_labels, fontsize=8)

# Remove spines (borders) for a cleaner look
for spine in ax.spines.values():
    spine.set_visible(False)

# Remove all grid lines and ticks to eliminate white lines
ax.tick_params(axis='both', which='both', length=0)

# Ensure no vertical gaps between columns
ax.set_xticks(np.arange(-0.5, 4, 1), minor=True)
ax.grid(False)  # Disable grid lines

# Adjust layout for minimal presentation
plt.tight_layout()

# Save the plot as a PDF file
pdf_filename = "cog_factor_loadings.pdf"
plt.savefig(pdf_filename, format='pdf', bbox_inches='tight', dpi=300)
plt.title("Estimated Bifactor Loading Matrix: pCAT-COG")
# Display the plot
plt.show()

print(f"Plot saved as {pdf_filename}")

In [ ]:
alphas